# 5-Year Pipeline: Part 4.2 - Global Model Training (Auto-Analysis)

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

print("Loading 5-Year Final Data for Global Model...")
df = pd.read_csv('5_year_final_data.csv')
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df['date'] = pd.to_datetime(df['date'])

df_clean = df.dropna(subset=['max_gain_7d', 'max_loss_7d']).copy()
df_clean = df_clean.sort_values('date').reset_index(drop=True)

n = len(df_clean)
train_end = int(n * 0.80)
train_df = df_clean.iloc[:train_end]
test_df = df_clean.iloc[train_end:]

leakage_cols = [
    'base_signal', 'signal_strength', 'entry_price', 'stop_loss', 'risk', 'target_1_2',
    'target_hit', 'sl_hit', 'timeout', 'outcome', 'days_to_outcome',
    'return_7d_after_signal', 'max_gain_7d', 'max_loss_7d', 'mfe', 'mae', 'Trade_Triggered',
    'max_r_achieved', 'date', 'symbol', 'sector', 'Stock Symbol', 'Sector', 'Trading_Day_Of_Week'
]
base_features = [c for c in df_clean.columns if c not in leakage_cols and pd.api.types.is_numeric_dtype(df_clean[c])]
base_features = [f for f in base_features if df_clean[f].isnull().mean() < 1.0]

X_train = train_df[base_features]
y_train_gain = train_df['max_gain_7d']
y_train_drop = train_df['max_loss_7d']

X_test = test_df[base_features]
y_test_gain = test_df['max_gain_7d']
y_test_drop = test_df['max_loss_7d']

imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()
X_train_s = scaler.fit_transform(imputer.fit_transform(X_train))
X_test_s = scaler.transform(imputer.transform(X_test))

Loading 5-Year Final Data for Global Model...


In [2]:
print("--- PHASE 1: TRAINING BASE MODELS ---")
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0]
}
base_xgb = xgb.XGBRegressor(random_state=42, objective='reg:squarederror')

print("Tuning Model 1: Max Gain Predictor (Base)...")
search_gain = RandomizedSearchCV(base_xgb, param_distributions=param_grid, n_iter=5, cv=3, scoring='neg_mean_absolute_error', random_state=42)
search_gain.fit(X_train_s, y_train_gain)
best_gain_base = search_gain.best_estimator_

print("Tuning Model 2: Max Drop Predictor (Base)...")
search_drop = RandomizedSearchCV(base_xgb, param_distributions=param_grid, n_iter=5, cv=3, scoring='neg_mean_absolute_error', random_state=42)
search_drop.fit(X_train_s, y_train_drop)
best_drop_base = search_drop.best_estimator_

os.makedirs('models', exist_ok=True)
joblib.dump(best_gain_base, 'models/global_gain_base.pkl')
joblib.dump(best_drop_base, 'models/global_drop_base.pkl')
joblib.dump(scaler, 'models/global_base_scaler.pkl')
joblib.dump(imputer, 'models/global_base_imputer.pkl')
with open('global_base_features.txt', 'w') as f:
    for feature in base_features:
        f.write(f"{feature}\n")

--- PHASE 1: TRAINING BASE MODELS ---
Tuning Model 1: Max Gain Predictor (Base)...


Tuning Model 2: Max Drop Predictor (Base)...


In [3]:
print("--- PHASE 2: AUTOMATIC FEATURE ANALYSIS ---")
imp_gain = best_gain_base.feature_importances_
imp_drop = best_drop_base.feature_importances_
avg_importances = (imp_gain + imp_drop) / 2.0
importance_df = pd.DataFrame({'Feature': base_features, 'Importance': avg_importances})

zero_imp = importance_df[importance_df['Importance'] == 0]['Feature'].tolist()

corr_matrix = pd.DataFrame(X_train_s, columns=base_features).corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = []
for col in upper.columns:
    high_corr = upper[col][upper[col] > 0.90].index.tolist()
    if high_corr:
        col_imp = importance_df[importance_df['Feature'] == col]['Importance'].values[0]
        for c in high_corr:
            c_imp = importance_df[importance_df['Feature'] == c]['Importance'].values[0]
            if col_imp < c_imp:
                to_drop_corr.append(col)
                break

features_to_remove = list(set(zero_imp + to_drop_corr))
refined_features = [f for f in base_features if f not in features_to_remove]

print(f"Removed {len(features_to_remove)} bad/redundant features.")
print(f"Remaining good features: {len(refined_features)}")

X_train_ref = train_df[refined_features]
X_test_ref = test_df[refined_features]

imputer_ref = SimpleImputer(strategy='median')
scaler_ref = StandardScaler()
X_train_s_ref = scaler_ref.fit_transform(imputer_ref.fit_transform(X_train_ref))
X_test_s_ref = scaler_ref.transform(imputer_ref.transform(X_test_ref))

--- PHASE 2: AUTOMATIC FEATURE ANALYSIS ---


Removed 48 bad/redundant features.
Remaining good features: 89


In [4]:
print("--- PHASE 3: TRAINING REFINED MODELS ---")
print("Tuning Model 1: Max Gain Predictor (Refined)...")
search_gain_ref = RandomizedSearchCV(base_xgb, param_distributions=param_grid, n_iter=5, cv=3, scoring='neg_mean_absolute_error', random_state=42)
search_gain_ref.fit(X_train_s_ref, y_train_gain)
best_gain_ref = search_gain_ref.best_estimator_

print("Tuning Model 2: Max Drop Predictor (Refined)...")
search_drop_ref = RandomizedSearchCV(base_xgb, param_distributions=param_grid, n_iter=5, cv=3, scoring='neg_mean_absolute_error', random_state=42)
search_drop_ref.fit(X_train_s_ref, y_train_drop)
best_drop_ref = search_drop_ref.best_estimator_

joblib.dump(best_gain_ref, 'models/5_year_global_gain_model.pkl')
joblib.dump(best_drop_ref, 'models/5_year_global_drop_model.pkl')
joblib.dump(scaler_ref, 'models/5_year_global_scaler.pkl')
joblib.dump(imputer_ref, 'models/5_year_global_imputer.pkl')

with open('global_features.txt', 'w') as f:
    for feature in refined_features:
        f.write(f"{feature}\n")

print("Saved all models successfully! Notebook 5.2 will evaluate and log results.")

--- PHASE 3: TRAINING REFINED MODELS ---
Tuning Model 1: Max Gain Predictor (Refined)...


Tuning Model 2: Max Drop Predictor (Refined)...


Saved all models successfully! Notebook 5.2 will evaluate and log results.
